In [ ]:
import os
from os.path import expanduser
home = expanduser("~/")

import sys
# sys.path.insert(0, '/global/u2/x/xshuang/gigalens-xh-dev/src')

# import sys
conda_env = sys.path[1]
del sys.path[1]

import os
# sys.path.append(f'{os.environ['HOME']}/gigalens_personal/gigalens/src')
sys.path.append(home+'/gigalens'+'/src')
sys.path.append(conda_env)
print(sys.path)



srcdir = os.path.join(home, "gigalens/src/")


In [ ]:
!pip show jax jaxlib

In [ ]:

import tensorflow_probability.substrates.jax as tfp

from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
from helpers import *
tfd = tfp.distributions
jax.devices()

In [ ]:
# * Load in data
systems_dir = "SystemSaves"

f = np.load(os.path.join(systems_dir, "100SystemsStandard80px.npz"))
keys = f.files
observed_imgs = jnp.array([f[key] for key in keys])




filename = os.path.join(systems_dir, '100SystemsStandardParams.yaml')
with open(filename, 'r') as file:
    true_params = params_lists_to_jax(yaml.safe_load(file))


In [ ]:
prior = make_default_prior()

In [ ]:
kernel = np.load('/global/homes/l/linusu/gigalens/src/gigalens/assets/psf.npy').astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=80, supersample=2, kernel=kernel)
phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
lens_sim = LensSimulator(phys_model, sim_config, bs=1)

# save_prefix = f'FitSaves/FitResults60px'

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
indices = [60]#list(range(len(observed_imgs)))

skip_indices = []#[4, 27]

for i in indices:
    print(i)
    img = observed_imgs[i]

    sys_true = jax.tree.map(lambda x : x[i], true_params)

    prob_model = ForwardProbModel(prior, img, background_rms=0.2, exp_time=100)
    model_seq = ModellingSequence(phys_model, prob_model, sim_config)
    best = jnp.array(prob_model.bij.inverse(sys_true))

    # def log_prob(params):
    #     lps = prob_model.log_prob(lens_sim, params)[0]
    #     return lps
    
    # map_hessian = jnp.squeeze(jax.hessian(log_prob)(best))
    # map_hess_cov = -jnp.linalg.inv(map_hessian)
    # map_hess_qz = tfp.distributions.MultivariateNormalFullCovariance(loc=jnp.squeeze(best), covariance_matrix=map_hess_cov)
    # cfg = PipelineConfig(steps=["MAP"], map_kwargs=dict(num_steps=1000, n_samples=5000))

    # cfg = PipelineConfig(steps=["MAP", "SVI", "HMC"], map_kwargs=dict(num_steps=1000, n_samples=2000),
    #     svi_kwargs=dict(num_steps=5000, n_vi=2000),
    #     hmc_kwargs=dict(n_hmc=64, num_results=250000, num_burnin_steps=40000, force_use_burnin=True), hmc_func=model_seq.HMC_alt_multi)

    cfg = PipelineConfig(steps=["MAP", "SVI", "HMC"], map_kwargs=dict(num_steps=350, n_samples=2000),
    svi_kwargs=dict(num_steps=10000, n_vi=1000),
    hmc_kwargs=dict(n_hmc=64, num_results=100000, num_burnin_steps=10000))
    
    # results = run_pipeline(model_seq, cfg)
    
    # results = simulate_system(img, prior, ModellingSequence, sim_config, phys_model,# map_kwargs=dict(num_steps=1000, n_samples=2000),
    #         svi_kwargs=dict(start=svi_start, num_steps=5000, n_vi=1000),
    #         hmc_kwargs=dict(n_hmc=64, num_results=5000, num_burnin_steps=10000, force_use_burnin=True), hmc_alt_multi = True)
    # results = {"map_best":map_best, "SVI_samples":SVI_samples, "HMC_samples":HMC_samples, "HMC_median":HMC_median, "HMC_img":HMC_img,
    #            "map_chisq_hist":map_chisq_hist, "svi_loss_hist":svi_loss_hist}
    
    # filename = f'{save_prefix}_{i}.pkl'
    # with open(filename, 'wb') as file:
    #     pickle.dump(results, file)
    

In [ ]:
# display_results(results, observed_imgs[i], lens_sim, true_params=index_params(true_params, i), model_seq=model_seq, make_cornerplot=True)

In [ ]:
results_dir = "sys60_converged_4e4_burnin"
results = {}
results["MAP"] = MAPResults.load(results_dir, model_seq)
results["SVI"] = SVIResults.load(results_dir, model_seq)
results["HMC"] = HMCResults.load(results_dir, model_seq)

In [ ]:
# results_dir = "sys60_converged_4e4_burnin"
# results["MAP"].save(results_dir)
# results["SVI"].save(results_dir)
# results["HMC"].save(results_dir)

In [ ]:
results['HMC'].HMC_samples_z.shape

In [ ]:
(4, 16, 250000, 22)
(16, 250000, 4, 22)

In [ ]:
# rhat_def = tfp.mcmc.potential_scale_reduction(jnp.transpose(results['HMC'].HMC_samples_z, (1,2,0,3)), independent_chain_ndims=2)
# print(rhat_def.shape)
# print(np.max(rhat_def))

In [ ]:
rhat_2103 = tfp.mcmc.potential_scale_reduction(jnp.transpose(results['HMC'].HMC_samples_z, (2,1,0,3)), independent_chain_ndims=2)
print(rhat_2103.shape)
print(np.max(rhat_2103))

In [ ]:
rhat_2013 = tfp.mcmc.potential_scale_reduction(jnp.transpose(results['HMC'].HMC_samples_z, (2,0,1,3)), independent_chain_ndims=2)
print(rhat_2013.shape)
print(np.max(rhat_2013))

In [ ]:
rhat_2 = tfp.mcmc.potential_scale_reduction(jnp.transpose(results['HMC'].HMC_samples_z.reshape(64, 250000, 22), (1, 0, 2)), independent_chain_ndims=1)
print(rhat_2.shape)
print(np.max(rhat_2))

In [ ]:
np.max(results['HMC'].HMC_rhat)

In [ ]:
all_samples = results['HMC'].HMC_samples_z.reshape(-1, 22)
mle_cov = jnp.cov(all_samples, rowvar=False)
hmc_qz = tfd.MultivariateNormalFullCovariance(
    loc=jnp.median(all_samples, axis=0),
    covariance_matrix=mle_cov,
)

In [ ]:
n_elbo = 1000
elbo_lens_sim = LensSimulator(phys_model, sim_config, bs=n_elbo)
def elbo(qz):
    z = qz.sample(n_elbo, seed=jax.random.PRNGKey(0))
    lps = qz.log_prob(z)
    return jnp.mean(lps - prob_model.log_prob(elbo_lens_sim, z)[0])

print(elbo(results['SVI'].qz))
print(elbo(hmc_qz))

In [ ]:
plt.plot(results['MAP'].MAP_chisq_hist)
plt.ylim(top=2)
plt.show()
results['MAP'].MAP_chisq_hist[-1]

In [ ]:
plt.plot(results['SVI'].SVI_loss_hist)
plt.show()

In [ ]:
# print(results['MAP'].time_taken)
results['SVI'].time_taken, results['HMC'].time_taken

In [ ]:
results['HMC'].HMC_samples[0][0]['e1'].shape

In [ ]:
# def cornerplot_posterior(raw_samples, fig=None, truth=None, overplots=None, color='black', truth_color='black', overplot_color='red', plot_params=None):
#     """
#     Create a cornerplot of a set of samples in the physical space.
#     Option to overplot a single point, such as the MAP best fit
#     Can also overplot a second point as crossed vertical and horizontal lines (most often the truth or median of the samples)
#     """
#     flat_samples = flatten_params_to_labeled_dict(raw_samples)
#     if plot_params is None:
#         plot_params = flat_samples.keys()
#         # flat_samples = {k:flat_samples[k] for k in plot_params}
        

#     if overplots is not None:
#         flat_overplots = flatten_params_to_labeled_dict(overplots)
#             #flat_overplots = {k:flat_overplots[k] for k in plot_params}
#         overplot_pts = np.squeeze(np.stack([flat_overplots[key] for key in plot_params]))

#     if truth is not None:
#         flat_truth = flatten_params_to_labeled_dict(truth)
#         # if plot_params is not None:
#         #     flat_truth = {k:flat_truth[k] for k in plot_params}
#         truth_overplot_pts = np.squeeze(np.stack([flat_truth[key] for key in plot_params]))
#     else:
#         truth_overplot_pts = None

#     samples = np.vstack([flat_samples[key] for key in plot_params]).T
#     histargs = {'density': True, 'color': color}
#     labels = [latex_label(label) for label in plot_params]
#     fig = corner.corner(samples, fig=fig, truths=truth_overplot_pts, truth_color=truth_color, 
#         show_titles=True, title_fmt='.3f',
#         labels=labels, hist_kwargs=histargs, color=color)

#     if overplots is not None:
#         corner.overplot_points(fig, overplot_pts[np.newaxis], marker='*', markersize=20, mfc=overplot_color, mec=overplot_color)
    
#     return fig

# def cornerplot_results(map_best, svi_samples=None, HMC_samples=None, true_params=None, hmc_median=None, plot_params=None):
#     """
#     Cornerplot of the results of the inference pipeline, including MAP, SVI, and HMC.
#     """

#     fig = cornerplot_posterior(svi_samples, truth=true_params, overplots=map_best, color='blue', truth_color='black', overplot_color='red', plot_params=plot_params)
#     cornerplot_posterior(HMC_samples, fig=fig, plot_params=plot_params)

def cornerplot_posterior(raw_samples, fig=None, truth=None, overplots=None, color='black', truth_color='black', overplot_color='red', plot_params=None):
    """
    Create a cornerplot of a set of samples in the physical space.
    Option to overplot a single point, such as the MAP best fit
    Can also overplot a second point as crossed vertical and horizontal lines (most often the truth or median of the samples)
    """
    flat_samples = flatten_params_to_labeled_dict(raw_samples)
    if plot_params is None:
        plot_params = flat_samples.keys()
        # flat_samples = {k:flat_samples[k] for k in plot_params}
        

    if overplots is not None:
        flat_overplots = flatten_params_to_labeled_dict(overplots)
            #flat_overplots = {k:flat_overplots[k] for k in plot_params}
        overplot_pts = np.squeeze(np.stack([flat_overplots[key] for key in plot_params]))

    if truth is not None:
        flat_truth = flatten_params_to_labeled_dict(truth)
        # if plot_params is not None:
        #     flat_truth = {k:flat_truth[k] for k in plot_params}
        truth_overplot_pts = np.squeeze(np.stack([flat_truth[key] for key in plot_params]))
    else:
        truth_overplot_pts = None

    samples = np.vstack([flat_samples[key] for key in plot_params]).T
    histargs = {'density': True, 'color': color}
    labels = [latex_label(label) for label in flat_samples.keys()]
    fig = corner.corner(samples, fig=fig, truths=truth_overplot_pts, truth_color=truth_color, 
        show_titles=True, title_fmt='.3f',
        labels=labels, hist_kwargs=histargs, color=color, label_kwargs=dict(fontsize=20), title_kwargs=dict(fontsize=15))

    if overplots is not None:
        corner.overplot_points(fig, overplot_pts[np.newaxis], marker='*', markersize=20, mfc=overplot_color, mec=overplot_color)
    
    return fig

def cornerplot_results(map_best, svi_samples=None, HMC_samples=None, true_params=None, hmc_median=None, plot_params=None, svi_label='SVI', hmc_label='HMC', legend_loc='upper right', legend_kwargs=None, truth_label='Truth', map_label='MAP'):
    """
    Cornerplot of the results of the inference pipeline, including MAP, SVI, and HMC.
    """

    svi_color = 'blue'
    hmc_color = 'black'

    if plot_params is None:
        plot_params = cornerplot_labels(map_best)

    fig = cornerplot_posterior(svi_samples, truth=true_params, overplots=map_best, color=svi_color, truth_color='black', overplot_color='red', plot_params=plot_params)
    cornerplot_posterior(HMC_samples, fig=fig, color=hmc_color, plot_params=plot_params)

    # Build a single consolidated legend
    handles = []
    if (svi_label is not None):
        handles.append(Patch(facecolor=svi_color, edgecolor='none', alpha=0.6, label=svi_label))
    if (hmc_label is not None):
        handles.append(Patch(facecolor=hmc_color, edgecolor='none', alpha=0.6, label=hmc_label))
    if (true_params is not None) and (truth_label is not None):
        handles.append(Line2D([0], [0], color='black', lw=4, label=truth_label))
    if (map_best is not None) and (map_label is not None):
        handles.append(Line2D([0], [0], marker='*', markersize=30, linestyle='none', markerfacecolor='red', markeredgecolor='red', label=map_label))

    if legend_kwargs is None:
        legend_kwargs = {}
    if 'fontsize' not in legend_kwargs:
        legend_kwargs['fontsize'] = 12
    prev_leg = getattr(fig, "_corner_legend_obj", None)
    if prev_leg is not None:
        try:
            prev_leg.remove()
        except Exception:
            pass
    new_leg = fig.legend(handles=handles, loc=legend_loc, frameon=False, **legend_kwargs)
    setattr(fig, "_corner_legend_obj", new_leg)

In [ ]:
def cornerplot_posterior_old(labels, raw_samples, fig=None, truth=None, overplots=None, color='black', truth_color='black', overplot_color='red'):
    """
    Create a cornerplot of the a set of samples in the physical space.
    Option to overplot a single point, such as the MAP best fit
    Can also overplot a second point as crossed vertical and horizontal lines (most often the truth or median of the samples)
    """
    tups = [(0, 0), (0, 1), (1, 0), (2, 0)]

    if overplots is not None:
        overplot_pts = []
        for (i, j) in tups:

            overplot_pts.extend((arr.item() for arr in overplots[i][j].values()))
        overplot_pts = np.array(overplot_pts)

    if truth is not None:
        truth_overplot_pts = []
        for (i,j) in tups:
            if i == 0 and j == 0:
                print(truth[i][j].keys())
                print(raw_samples[i][j].keys())
            truth_overplot_pts.extend((arr.item() for arr in truth[i][j].values()))
        truth_overplot_pts = np.array(truth_overplot_pts)
    else:
        truth_overplot_pts = None

    samples = np.vstack([np.array(list(raw_samples[i][j].values())) for i, j in tups]).T
    histargs = {'density': True, 'color': color}
    
    fig = corner.corner(samples, fig=fig, truths=truth_overplot_pts, truth_color=truth_color, 
        show_titles=True, title_fmt='.3f',
        labels=labels, hist_kwargs=histargs, color=color)

    if overplots is not None:
        corner.overplot_points(fig, overplot_pts[np.newaxis], marker='*', markersize=12, mfc=overplot_color, mec=overplot_color)
    
    return fig

def cornerplot_results_old(map_best, svi_samples=None, HMC_samples=None, true_params=None, hmc_median=None):
    """
    Cornerplot of the results of the inference pipeline, including MAP, SVI, and HMC.
    """
    labels = cornerplot_labels(map_best)

    fig = cornerplot_posterior_old(labels, svi_samples, truth=true_params, overplots=map_best, color='blue', truth_color='green', overplot_color='red')
    cornerplot_posterior_old(labels, HMC_samples, fig=fig, truth=hmc_median)

In [ ]:
def plot_image_paper(fig, ax, img, extent=None, title=None, residual=False, colorbar=True):
    """
    Plot an image using my chosen standards for coloring, 
    which changes depending on whether the image is a residual or not.
    """
    if not residual:
        #* Meaning actual lensing image
        # cnorm = matplotlib.colors.Normalize(vmin=0)
        # Use LogNorm for logarithmic scaling with inferno colormap
        cnorm = matplotlib.colors.LogNorm(vmin=max(img.min(), 1e0), vmax=img.max())
        cmap = 'magma'
    else:
        #* Meaning residual image
        cnorm = matplotlib.colors.CenteredNorm()
        cmap = 'bwr'
    
    if colorbar:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)


    im = ax.imshow(img, cmap=cmap, norm=cnorm, extent=extent, origin='lower')
    if colorbar:
        fig.colorbar(im, cax=cax)
    if title is not None:
        ax.set_title(title)
    if extent is not None:
        ax.set_xlim((extent[0], extent[1]))
        ax.set_ylim((extent[2], extent[3]))
    ax.axis('off')

def plot_image_results_paper(fig, axs, true_img, lens_sim=None, predicted_params=None, 
                       predicted_img=None, resimulate=True, display_true_chisq=False, true_params=None, prefix="",
                       model_seq=None):
    """
    Plot the results of a lensing fit. Given a set of predicted parameters, compare the predicted image to the true image.
    Displays normalized residuals, and a histogram of the residuals to check that they are gaussian noise
    """
    if resimulate:
        if lens_sim is None:
            raise ValueError("lens_sim must be provided if resimulate is True")
        predicted_img = lens_sim.simulate(predicted_params)
    elif predicted_img is None:
        raise ValueError("predicted_img must be provided if resimulate is False")

    if display_true_chisq:
        true_chisq = get_chisq(true_img, lens_sim.simulate(true_params))
    
    noise_map = get_noise_image(true_img, 0.2, 100)

    residual = (true_img - predicted_img)/noise_map

    chisq = np.sum(np.square(residual))
    dof = true_img.shape[0]*true_img.shape[1] - 22 #! Change if number of params changes
    #! Do I want to do sqrt curve cmap for the images?\
    numPix = model_seq.sim_config.num_pix
    deltaPix = model_seq.sim_config.delta_pix
    extent = (-numPix/2*deltaPix, numPix/2*deltaPix, -numPix/2*deltaPix, numPix/2*deltaPix)
    
    plot_image_paper(fig, axs[0], true_img, extent=extent,
               title=f"Observed Image" + (f"(Red Chisq:{true_chisq/dof:.3f})" if display_true_chisq else ""))
    plot_image_paper(fig, axs[1], predicted_img, extent=extent, title=rf"{prefix} ($\tilde{{\chi}}^2 = {chisq/dof:.3f}$)")
    plot_image_paper(fig, axs[2], residual, extent=extent, title=f"Normalized Residual", residual=True)
    plot_image_paper(fig, axs[3], lens_sim.simulate([[], [], predicted_params[2]]), extent=extent, title=f"Source Plane")
    add_caustics(axs[3], predicted_params, model_seq)
    

    if display_true_chisq:
        print("True Chisq", true_chisq)
        print("Model Fit Chisq", chisq)

In [ ]:
fig, axs = plt.subplots(2, 2)
axs = axs.flatten()
fig.set_size_inches(6,6)
plot_image_results_paper(fig, axs, img, prefix="HMC Centroid",
                   lens_sim=lens_sim, predicted_params=results['HMC'].HMC_median, 
                   resimulate=True, true_params=sys_true, model_seq=model_seq)

In [ ]:
n_samp = results['HMC'].HMC_samples[0][0]['e1'].shape[0]
rand_idx = np.random.choice(np.arange(n_samp), size=(20000,), replace=True)
# cornerplot_results(sys_true, results['SVI'].SVI_samples, jax.tree.map(lambda x: x[rand_idx], results['HMC'].HMC_samples), 
#                    true_params=sys_true, hmc_median=results['HMC'].HMC_median)
# labels = cornerplot_labels(sys_true)
cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx], results['HMC'].HMC_samples), truth=sys_true, overplots=results['MAP'].MAP_best)
plt.show()

In [ ]:
z = results['SVI'].qz.sample(10000, seed=jax.random.PRNGKey(0))
SVI_samples = prob_model.bij.forward(list(z.T))
cornerplot_results(results['MAP'].MAP_best, SVI_samples, jax.tree.map(lambda x: x[rand_idx], results['HMC'].HMC_samples), 
                   true_params=sys_true, hmc_median=results['HMC'].HMC_median, legend_kwargs={'fontsize':60}, legend_loc=(0.2, 0.87))

In [ ]:
# print(jax.tree.structure(results['MAP'].MAP_best))
# print(jax.tree.structure(sys_true))
# print(jax.tree.structure(results['HMC'].HMC_samples))

tups = [(0, 0), (0, 1), (1, 0), (2, 0)]
label_prefixes = ['', '', 'lens_', 'src_']

flat_map = []
for (i, j), label_prefix in zip(tups, label_prefixes):
    flat_map.extend([label_prefix + key for key in results['MAP'].MAP_best[i][j].keys()])

flat_svi = []
for (i, j), label_prefix in zip(tups, label_prefixes):
    flat_svi.extend([label_prefix + key for key in SVI_samples[i][j].keys()])

flat_hmc = []
for (i, j), label_prefix in zip(tups, label_prefixes):
    flat_hmc.extend([label_prefix + key for key in results['HMC'].HMC_samples[i][j].keys()])

flat_true = []
for (i, j), label_prefix in zip(tups, label_prefixes):
    flat_true.extend([label_prefix + key for key in sys_true[i][j].keys()])

print("MAP:",flat_map)
print("SVI:",flat_svi)
print("HMC:",flat_hmc)
print("TRU:",flat_true)

In [ ]:
plot_params = ["theta_E", "gamma", "e1", "e2", "src_e1", "src_e2"]
cornerplot_results(results['MAP'].MAP_best, SVI_samples, jax.tree.map(lambda x: x[rand_idx], results['HMC'].HMC_samples), 
                   true_params=sys_true, hmc_median=results['HMC'].HMC_median, plot_params=plot_params)

In [ ]:
# results["MAP"] = MAPResults(best, [], 0, model_seq)
display_results(results, observed_imgs[i], lens_sim, true_params=index_params(true_params, i), model_seq=model_seq, make_cornerplot=False)

In [ ]:
fig, axs = plt.subplots(1, 4)
fig.set_size_inches(12,3)
plot_image_results(fig, axs, observed_imgs[i], prefix="SVI",
                   lens_sim=lens_sim, predicted_params=prob_model.bij.forward(list(results['SVI'].qz.mean())), 
                   resimulate=True, true_params=sys_true, plot_caustics=False, model_seq=model_seq)
plt.show()

In [ ]:
chains = results["HMC"].HMC_samples_z.reshape(64, 250000, 22)
cdiff = np.diff(chains, axis=1)
stuck_locs = np.all(np.isclose(cdiff, 0), axis=2)

# print(np.sum(stuck_locs, axis=1))
from itertools import groupby

max_stuck = [np.max([len(list(g))for k, g in groupby(stuck_locs[i])]) for i in range(chains.shape[0])]
# np.sort(stuck_times)

In [ ]:
np.sort(max_stuck)

In [ ]:
unstuck_chains = chains[np.array(max_stuck) < 10000]
unstuck_samples = unstuck_chains.reshape(-1, 22)
unstuck_chains.shape

In [ ]:
rhat_2 = tfp.mcmc.potential_scale_reduction(jnp.transpose(unstuck_chains, (1, 0, 2)), independent_chain_ndims=1)
print(rhat_2.shape)
print(np.max(rhat_2))

In [ ]:
n_samp = unstuck_chains.shape[0] * unstuck_chains.shape[1]
# rand_idx = np.random.choice(np.arange(n_samp), size=(250000,), replace=True)

In [ ]:

cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx], prob_model.bij.forward(list(unstuck_samples.T))), truth=sys_true, overplots=results['MAP'].MAP_best, truth_color='black')
plt.show()

In [ ]:
a = prob_model.bij.forward(list(np.moveaxis(results["HMC"].HMC_samples_z, -1, 0)))


In [ ]:
# plt.plot(a[0][0]['e2'].reshape(-1, 250000)[0, 55000:85000])
stuck_chain = np.argmax(max_stuck)
dot_params = jax.tree.map(lambda x : x.reshape(-1, 250000)[stuck_chain, 10], a)
print("Stuck Chain Log Prob:", prob_model.log_prob(lens_sim, jnp.array(prob_model.bij.inverse(dot_params)))[0])
print("Truth Log Prob:", prob_model.log_prob(lens_sim, jnp.array(prob_model.bij.inverse(sys_true)))[0])
print("MAP Samp Log Prob:", prob_model.log_prob(lens_sim, jnp.squeeze(jnp.array(prob_model.bij.inverse(results['MAP'].MAP_best))))[0])
print("HMC Median Log Prob:", prob_model.log_prob(lens_sim, jnp.array(prob_model.bij.inverse(results['HMC'].HMC_median)))[0])

In [ ]:
fig, axs = plt.subplots(1, 4)
fig.set_size_inches(12,3)
plot_image_results(fig, axs, observed_imgs[i], prefix="Stuck",
                   lens_sim=lens_sim, predicted_params=dot_params, 
                   resimulate=True, true_params=sys_true, plot_caustics=True, model_seq=model_seq)
plt.show()
fig, axs = plt.subplots(1, 4)
fig.set_size_inches(12,3)
plot_image_results(fig, axs, observed_imgs[i], prefix="HMC",
                   lens_sim=lens_sim, predicted_params=results['HMC'].HMC_median, 
                   resimulate=True, true_params=sys_true, plot_caustics=True, model_seq=model_seq)
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec

def plot_trace_and_hist(fig, spec, all_data, title=""):
    
    nested_gs = spec.subgridspec(1, 2, width_ratios=[4, 1], wspace=0.05)
    ax_trace = fig.add_subplot(nested_gs[0, 0])
    ax_hist = fig.add_subplot(nested_gs[0, 1], sharey=ax_trace)
    
    num_chains, num_samples = all_data.shape
    
    colors = plt.get_cmap('tab10', num_chains)
    
    for i in range(num_chains):
        chain_data = all_data[i]
        color = colors(i)

        ax_trace.plot(np.arange(0, 250000, 100), chain_data[::100], color=color, alpha=0.7, linewidth=1)
        ax_hist.hist(chain_data, 
                     bins=50, 
                     orientation='horizontal', 
                     density=True, 
                     color=color, 
                     alpha=0.7, 
                     histtype='step', 
                     linewidth=1.5)
                     
    
    # Style trace plot
    ax_trace.set_title(title, loc='left', fontsize='medium')
    ax_trace.set_xlabel("Iterations")
    ax_trace.set_ylabel("Parameter Value")
    ax_trace.grid(True, linestyle='--', alpha=0.6)
    ax_trace.set_xlim(0, num_samples) # Ensure x-axis starts at 0
    
    # Style histogram plot
    ax_hist.set_xticks([]) # Remove x-axis ticks (density values)
    plt.setp(ax_hist.get_yticklabels(), visible=False) # Hide y-axis labels
    ax_hist.grid(True, linestyle='--', alpha=0.6)
    
    return ax_trace, ax_hist

fig = plt.figure(figsize=(20, 20))
main_gs = fig.add_gridspec(4, 4, hspace=0.5) # Add vertical space

# --- 2. Loop and create each plot ---
for i in range(4):
    for j in range(4):

        k = 4*i + j
        step = 64//16
        
        # Generate some fake data for this plot
        # Each plot will have a different mean and scale
        data = a[0][0]['e2'].reshape(-1, 250000)[step*k:step*(k+1)]
        
        # Call our function to create the plot in the correct grid location
        plot_trace_and_hist(fig, 
                            main_gs[i, j], 
                            data, 
                            title=f"e2 Chains {step*k+1}-{step*(k+1)}")
                        
# --- 3. Show the final figure ---
# fig.suptitle("Tiled Trace & Histogram Plots", fontsize=16, y=0.99)
# plt.tight_layout(rect=[0, 0, 1, 1]) # Adjust for suptitle

# Save or show the plot
# plt.savefig("tiled_plots.png", dpi=150)
plt.show()

In [ ]:
fig, axs = plt.subplots(4,4)
fig.set_size_inches(20, 20)
axs = axs.flatten()

for i, ax in enumerate(axs):
    ax.plot(, linewidth=0.25)
plt.show()

In [ ]:
# results_dir = "sys60_converged"
# results["MAP"].save(results_dir)
# results["SVI"].save(results_dir)
# results["HMC"].save(results_dir)

In [ ]:
ev = jnp.linalg.eigvalsh(c)
jnp.min(ev), jnp.max(ev)

In [ ]:
# refit_idx = 56
# res = simulate_system(observed_imgs[refit_idx], prior, sim_config, 
#                       phys_model, lens_sim, start_svi_from_truth=True, 
#                       true_params=index_params(true_params, refit_idx), map_steps=350, n_vi=10000)


# with open(f'{save_prefix}_{refit_idx}_SVITruthVI10000.pkl', 'wb') as file:
#     pickle.dump(res, file)

In [ ]:
# results["MAP"] = MAPResults(best, [], 0, model_seq)
display_results(results, observed_imgs[i], lens_sim, true_params=index_params(true_params, i), model_seq=model_seq)

In [ ]:

i = 54
filename = f'FitSaves/FitResults_{i}.pkl'
with open(filename, 'rb') as file:
    r_original = pickle.load(file)

filename = f'FitSaves/ReFitResults_{i}.pkl'
with open(filename, 'rb') as file:
    r_refit = pickle.load(file)

filename = f'FitSaves/ReFitResults_{i}_MAP5600.pkl'
with open(filename, 'rb') as file:
    r_refit2 = pickle.load(file)

plt.title("SVI Curves with different MAP starting points")
plt.plot(r_original['svi_loss_hist'], label=f"350 MAP Steps")
plt.plot(r_refit['svi_loss_hist'], label=f"1400 MAP Steps")
plt.plot(r_refit2['svi_loss_hist'], label=f"5600 MAP Steps")
plt.legend()
plt.yscale('log')
plt.show()


In [ ]:
schedule_fn = optax.polynomial_schedule(init_value=-1e-6, end_value=-3e-3,
                                          power=2, transition_steps=300)

dummy_x = np.arange(1500)
plt.plot(dummy_x, schedule_fn(dummy_x))
plt.title("SVI Schedule Function")
plt.show()

In [ ]:
from scipy.signal import correlate2d
autocorr = correlate2d(observed_imgs[0], norm_residuals[0], mode='full', boundary='symm')
autocorr /= np.max(autocorr)  # normalize
plt.imshow(autocorr)